# Bayesian Learning of GP-LVM

$$
$$

<!-- CIP-0012 Track A: no-op outside HTML (overridden in talk-macros-html.gpp). -->
<!-- Class-test timer: no-op outside slides HTML (overridden in talk-macros-slides-html.gpp). -->
<!-- CIP-0016: mdpp expands \diagramcredit before gpp; leftover calls are empty. -->
<!-- Do not edit this file locally. -->
<!-- Do not edit this file locally. -->
<!---->
<!-- CIP-0012 Track A: content script load. Once-guard with alphanumeric
     names stays in the surrounding ifndef wrapper (GPP ifndef limitation). -->
<!-- Do not edit this file locally. -->
<!-- Do not edit this file locally. -->
<!-- The last names to be defined. Should be defined entirely in terms of macros from above-->
<!--

-->

## Gaussian Process Summer School notebook setup

<span class="editsection-bracket"
style="">\[</span><span class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_gpss/includes/gpss-notebook-setup.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_gpss/includes/gpss-notebook-setup.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

We install packages used across the notebooks for data loading, plotting
helpers, and Gaussian process modelling.

In [ ]:
import importlib.util

In [ ]:
def install_command(imp):
    import shutil
    import sys
    packages = {
        'daft': 'daft-pgm',
        'mlai': 'git+https://github.com/lawrennd/mlai.git',
        'pods': 'pods',
        'PyDeepGP': 'git+https://github.com/SheffieldML/PyDeepGP.git',
        'notutils': 'git+https://github.com/lawrennd/notutils.git',
        'pods' : 'git+https://github.com/lawrennd/ods.git',
        'lamd': 'lamd',
        'GPy': 'gpy>=1.15',
    'emukit': 'emukit',
    }

    def _installer():
        if shutil.which('uv'):
            return 'uv pip install --python "' + sys.executable + '"'
        return '"' + sys.executable + '" -m pip install'

    def _ver_tuple(v):
        parts = []
        for p in v.split('.')[:3]:
            try:
                parts.append(int(''.join(ch for ch in p if ch.isdigit()) or '0'))
            except ValueError:
                parts.append(0)
        while len(parts) < 3:
            parts.append(0)
        return tuple(parts)

    def _below_floor(spec):
        if '>=' not in spec:
            return False
        name, floor = spec.split('>=', 1)
        name = name.strip()
        floor = floor.strip().split(',')[0].strip()
        try:
            import importlib.metadata as md
            cur = md.version(name)
        except Exception:
            return True
        return _ver_tuple(cur) < _ver_tuple(floor)

    spec = packages[imp] if imp in packages else imp
    missing = importlib.util.find_spec(imp) is None
    if missing or _below_floor(spec):
        return _installer() + ' "' + spec + '"'
    return 'echo "Package ' + imp + ' is already installed."'

In [ ]:
cmd = install_command('pods')

In [ ]:
%system {cmd}

In [ ]:
cmd = install_command('notutils')

In [ ]:
%system {cmd}

In [ ]:
cmd = install_command('mlai')

In [ ]:
%system {cmd}

In [ ]:
import matplotlib.pyplot as plt
import shutil

if shutil.which('latex') is None:
    plt.rcParams['text.usetex'] = False
else:
    plt.rcParams['text.usetex'] = True
    plt.rcParams['text.latex.preamble'] = (
        r'\usepackage{amsmath}\usepackage{amssymb}\usepackage{mathrsfs}'
    )

plt.rcParams.update({'font.size': 22})

<!--setupplotcode{import seaborn as sns
sns.set_style('darkgrid')
sns.set_context('paper')
sns.set_palette('colorblind')}-->

In [ ]:
cmd = install_command('GPy')

In [ ]:
%system {cmd}

## GPy: A Gaussian Process Framework in Python

<span class="editsection-bracket"
style="">\[</span><span class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_software/includes/gpy-software.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_software/includes/gpy-software.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

Gaussian processes are a flexible tool for non-parametric analysis with
uncertainty. The GPy software was started in Sheffield to provide a easy
to use interface to GPs. One which allowed the user to focus on the
modelling rather than the mathematics.

<img class="" src="https://mlatcl.github.io/gpss/slides/diagrams/gp/gpy.png" style="width:70%; height:auto; display:block;">

Figure: <i>GPy is a BSD licensed software code base for implementing
Gaussian process models in Python. It is designed for teaching and
modelling. We welcome contributions which can be made through the GitHub
repository <https://github.com/SheffieldML/GPy></i>

GPy is a BSD licensed software code base for implementing Gaussian
process models in python. This allows GPs to be combined with a wide
variety of software libraries.

The software itself is available on
[GitHub](https://github.com/SheffieldML/GPy) and the team welcomes
contributions.

The aim for GPy is to be a probabilistic-style programming language,
i.e., you specify the model rather than the algorithm. As well as a
large range of covariance functions the software allows for non-Gaussian
likelihoods, multivariate outputs, dimensionality reduction and
approximations for larger data sets.

The documentation for GPy can be found
[here](https://gpy.readthedocs.io/en/latest/).

-   GP-LVM Provides probabilistic non-linear dimensionality reduction.
-   How to select the dimensionality?
-   Need to estimate marginal likelihood.
-   In standard GP-LVM it increases with increasing $q$.

**Bayesian GP-LVM**

-   Start with a standard GP-LVM.
-   Apply standard latent variable approach:
    -   Define Gaussian prior over , $\mathbf{Z}$.
    -   Integrate out .
    -   Unfortunately integration is intractable.

      \only<1->{\begin{tikzpicture}
          
          % Define nodes
          \node[obs]                               (Y) {$\dataMatrix$};
          \node[latent, above=of Y] (X) {$\latentMatrix$};
          \node[const, right=1cm of Y]            (sigma) {$\dataStd^2$};
          
          % Connect the nodes
          \edge {X,sigma} {Y} ; %
          
          % Plates
          % \plate {YX} {(Y)} {$i=1\dots\numData$} ;
          % \plate {} {(W)(Y)(YX.north west)(YX.south west)} {$j=1\dots\dataDim$} ;
          
        \end{tikzpicture}
      }
    <center>
      {\scriptsize \only<1->{\[
          p\left(\dataMatrix|\latentMatrix\right)=\prod_{j=1}^{\dataDim}\gaussianDist{\dataVector_{:,j}}{\zerosVector}{\kernelMatrix}
          \]
        }\only<3->{\[
          p\left(\latentMatrix\right)=\prod_{j=1}^{\latentDim}\gaussianDist{\latentVector_{:,j}}{\zerosVector}{\alpha_i^{-2}\eye}
          \]
        }\only<4->{\[ 
          p\left(\dataMatrix|\boldsymbol{\alpha}\right)= ??
          \]
        }
      }</center>

## Standard Variational Approach Fails

<span class="editsection-bracket"
style="">\[</span><span class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_gplvm/includes/variational-bayes-gplvm-long.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_gplvm/includes/variational-bayes-gplvm-long.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

-   Standard variational bound has the form: $$
    \mathscr{L}= \left\langle\log p(\mathbf{ y}|\mathbf{Z})\right\rangle_{q(\mathbf{Z})} + \text{KL}\left( q(\mathbf{Z})\,\|\,p(\mathbf{Z}) \right)
    $$

The standard variational approach would require the expectation of
$\log p(\mathbf{ y}|\mathbf{Z})$ under $q(\mathbf{Z})$. $$
  \begin{align}
  \log p(\mathbf{ y}|\mathbf{Z}) = & -\frac{1}{2}\mathbf{ y}^\top\left(\mathbf{K}_{\mathbf{ f}, \mathbf{ f}}+\sigma^2\mathbf{I}\right)^{-1}\mathbf{ y}\\ & -\frac{1}{2}\log \det{\mathbf{K}_{\mathbf{ f}, \mathbf{ f}}+\sigma^2 \mathbf{I}} -\frac{n}{2}\log 2\pi
  \end{align}
  $$ But this is extremely difficult to compute because
$\mathbf{K}_{\mathbf{ f}, \mathbf{ f}}$ is dependent on $\mathbf{Z}$ and
it appears in the inverse.

## Variational Bayesian GP-LVM

The alternative approach is to consider the collapsed variational bound
(used for low rank (sparse is a misnomer) Gaussian process
approximations. $$
    p(\mathbf{ y})\geq \prod_{i=1}^nc_i \int \mathscr{N}\left(\mathbf{ y}|\left\langle\mathbf{ f}\right\rangle,\sigma^2\mathbf{I}\right)p(\mathbf{ u}) \text{d}\mathbf{ u}
  $$ $$
    p(\mathbf{ y}|\mathbf{Z})\geq \prod_{i=1}^nc_i \int \mathscr{N}\left(\mathbf{ y}|\left\langle\mathbf{ f}\right\rangle_{p(\mathbf{ f}|\mathbf{ u}, \mathbf{Z})},\sigma^2\mathbf{I}\right)p(\mathbf{ u}) \text{d}\mathbf{ u}
  $$ $$
      \int p(\mathbf{ y}|\mathbf{Z})p(\mathbf{Z}) \text{d}\mathbf{Z}\geq \int \prod_{i=1}^nc_i \mathscr{N}\left(\mathbf{ y}|\left\langle\mathbf{ f}\right\rangle_{p(\mathbf{ f}|\mathbf{ u}, \mathbf{Z})},\sigma^2\mathbf{I}\right) p(\mathbf{Z})\text{d}\mathbf{Z}p(\mathbf{ u}) \text{d}\mathbf{ u}
  $$

To integrate across $\mathbf{Z}$ we apply the lower bound to the inner
integral. $$
    \begin{align}
    \int \prod_{i=1}^nc_i \mathscr{N}\left(\mathbf{ y}|\left\langle\mathbf{ f}\right\rangle_{p(\mathbf{ f}|\mathbf{ u}, \mathbf{Z})},\sigma^2\mathbf{I}\right) p(\mathbf{Z})\text{d}\mathbf{Z}\geq & \left\langle\sum_{i=1}^n\log  c_i\right\rangle_{q(\mathbf{Z})}\\ & +\left\langle\log\mathscr{N}\left(\mathbf{ y}|\left\langle\mathbf{ f}\right\rangle_{p(\mathbf{ f}|\mathbf{ u}, \mathbf{Z})},\sigma^2\mathbf{I}\right)\right\rangle_{q(\mathbf{Z})}\\& + \text{KL}\left( q(\mathbf{Z})\,\|\,p(\mathbf{Z}) \right)    
    \end{align}
  $$ \* Which is analytically tractable for Gaussian $q(\mathbf{Z})$ and
some covariance functions.

-   Need expectations under $q(\mathbf{Z})$ of: $$
    \log c_i = \frac{1}{2\sigma^2} \left[k_{i, i} - \mathbf{ k}_{i, \mathbf{ u}}^\top \mathbf{K}_{\mathbf{ u}, \mathbf{ u}}^{-1} \mathbf{ k}_{i, \mathbf{ u}}\right]
    $$ and $$
    \log \mathscr{N}\left(\mathbf{ y}|\left\langle\mathbf{ f}\right\rangle_{p(\mathbf{ f}|\mathbf{ u},\mathbf{Y})},\sigma^2\mathbf{I}\right) = -\frac{1}{2}\log 2\pi\sigma^2 - \frac{1}{2\sigma^2}\left(y_i - \mathbf{K}_{\mathbf{ f}, \mathbf{ u}}\mathbf{K}_{\mathbf{ u},\mathbf{ u}}^{-1}\mathbf{ u}\right)^2
    $$

-   This requires the expectations $$
    \left\langle\mathbf{K}_{\mathbf{ f},\mathbf{ u}}\right\rangle_{q(\mathbf{Z})}
    $$ and $$
    \left\langle\mathbf{K}_{\mathbf{ f},\mathbf{ u}}\mathbf{K}_{\mathbf{ u},\mathbf{ u}}^{-1}\mathbf{K}_{\mathbf{ u},\mathbf{ f}}\right\rangle_{q(\mathbf{Z})}
    $$ which can be computed analytically for some covariance functions
    (Damianou et al., 2016) or through sampling (Damianou, 2015;
    Salimbeni and Deisenroth, 2017).

## Manifold Relevance Determination

<span class="editsection-bracket"
style="">\[</span><span class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_gplvm/includes/mrd-gplvm.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_gplvm/includes/mrd-gplvm.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

# Modeling Multiple ‘Views’

<span class="editsection-bracket"
style="">\[</span><span class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_gplvm/includes/shared-intro.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_gplvm/includes/shared-intro.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

-   Single space to model correlations between two different data
    sources, e.g., images & text, image & pose.

-   Shared latent spaces: Shon et al. (n.d.);Navaratnam et al. (2007);Ek
    et al. (2008b)

-   Effective when the \`views’ are correlated.

-   But not all information is shared between both \`views’.

-   PCA applied to concatenated data vs CCA applied to data.

# Shared-Private Factorization

-   In real scenarios, the \`views’ are neither fully independent, nor
    fully correlated.

-   Shared models

    -   either allow information relevant to a single view to be mixed
        in the shared signal,
    -   or are unable to model such private information.

-   Solution: Model shared and private information Virtanen et al.
    (n.d.),Ek et al. (2008a),Leen and Fyfe (2006),Klami and Kaski
    (n.d.),Klami and Kaski (2008),Tucker (1958)

-   Probabilistic CCA is case when dimensionality of $\mathbf{Z}$
    matches $\mathbf{Y}^{(i)}$ (cf Inter Battery Factor Analysis Tucker
    (1958)).

<!--frame failure start-->

# Manifold Relevance Determination

<span class="editsection-bracket"
style="">\[</span><span class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_gplvm/includes/mrd-intro.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_gplvm/includes/mrd-intro.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

    \def\layersep{2cm}
    \begin{center}
      \begin{tikzpicture}[node distance=\layersep]

= \[text width=4em, text centered\] % Draw the input layer nodes / in
{1,…,8} % This is the same as writing / in {1/1,2/2,3/3,4/4} (Y-) at (,
0) {$y_\x$};

    % Draw the hidden layer nodes
    \foreach \name / \x in {1,...,6}
        \path[xshift=1cm]
            node[latent] (X-\name) at (\x cm, \layersep) {$\latentScalar_\x$};

    % Connect every node in the latent layer with every node in the
    % data layer.
    \foreach \source in {1,...,6}
        \foreach \dest in {1,...,8}
            \draw[->] (X-\source) -- (Y-\dest);



    %  Annotate the layers
    \node[annot,left of=X-1, node distance=1cm] (ls) {Latent space};
    \node[annot,left of=Y-1, node distance=1cm] (ds) {Data space};

\end{tikzpicture} \end{center}

<!--frame failure end-->
<!--frame failure start-->

# Shared GP-LVM

    \def\layersep{2cm}
    \begin{center}
      \begin{tikzpicture}[node distance=\layersep]

= \[text width=4em, text centered\] % Draw the input layer nodes / in
{1,…,4} % This is the same as writing / in {1/1,2/2,3/3,4/4} (Y-) at (,
0) {$y^{(1)}_\x$};

    \foreach \name / \x in {1,...,4}
    % This is the same as writing \foreach \name / \x in {1/1,2/2,3/3,4/4}
        \node[obs] (Z-\name) at (\x+5, 0) {$\dataScalar^{(2)}_\x$};

    % Draw the hidden layer nodes
    \foreach \name / \x in {1,...,6}
        \path[xshift=2cm]
            node[latent] (X-\name) at (\x cm, \layersep) {$\latentScalar_\x$};

    % Connect every node in the latent layer with every node in the
    % data layer.
    \foreach \source in {1,...,6}
        \foreach \dest in {1,...,4}
            \draw[->] (X-\source) -- (Y-\dest);

    \foreach \source in {1,...,6}
        \foreach \dest in {1,...,4}
            \draw[->] (X-\source) -- (Z-\dest);


    %  Annotate the layers
    \node[annot,left of=X-1, node distance=1cm] (ls) {Latent space};
    \node[annot,left of=Y-1, node distance=1cm] (ds) {Data space};

\end{tikzpicture} Separate ARD parameters for mappings to
$\mathbf{Y}^{(1)}$ and $\mathbf{Y}^{(2)}$. \end{center}

# Manifold Relevance Determination Results

<span class="editsection-bracket"
style="">\[</span><span class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_gplvm/includes/mrd-results.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_gplvm/includes/mrd-results.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

<img class="" src="https://mlatcl.github.io/gpss/slides/diagrams/gplvm/andreas-deep-talk-0.png" style="width:90%; height:auto; display:block;">

Figure: <i>The Yale Faces data set shows different people under
different lighting conditions</i>

<img class="" src="https://mlatcl.github.io/gpss/slides/diagrams/gplvm/andreas-deep-talk-1.png" style="width:90%; height:auto; display:block;">

Figure: <i>ARD Demonstrates not all of the latent space is used.</i>

<img class="" src="https://mlatcl.github.io/gpss/slides/diagrams/gplvm/andreas-deep-talk-2.png" style="width:90%; height:auto; display:block;">

Figure: <i>Other applications include inferring pose from silhouette</i>

In [ ]:
from IPython.lib.display import YouTubeVideo
YouTubeVideo('UvLI8o8z4IU')

Figure: <i>A short video description of the Manifold Relevance
Determination method as published at ICML 2012</i>

## Getting Started and Downloading Data

<span class="editsection-bracket"
style="">\[</span><span class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_gplvm/includes/gplvm-tutorial-data.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_gplvm/includes/gplvm-tutorial-data.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

In [ ]:
import numpy as np
import GPy
import string

The following code is for plotting and to prepare the bigger models for
later useage. If you are interested, you can have a look, but this is
not essential.

In [ ]:
from matplotlib import pyplot as plt
import mlai.plot as plot
import mlai

In [ ]:
colors = ["#3FCC94", "#DD4F23", "#C6D63B", "#D44271", 
          "#E4A42C", "#4F9139", "#6DDA4C", "#85831F", 
          "#B36A29", "#CF4E4A"]

In [ ]:
def plot_model(X, which_dims, labels):
    fig, ax = plt.subplots(figsize=plot.big_figsize)
    X = X[:,which_dims]
    ulabs = []
    for lab in labels:
        if not lab in ulabs:
            ulabs.append(lab)
            pass
        pass
    for i, lab in enumerate(ulabs):
        ax.scatter(*X[labels==lab].T,marker='o',color=colors[i],label=lab)
        pass
    pass

For this lab, we’ll use a data set containing all handwritten digits
from $0 \cdots 9$ handwritten, provided by de Campos et al. (2009). We
will only use some of the digits for the demonstrations in this lab
class, but you can edit the code below to select different subsets of
the digit data as you wish.

In [ ]:
import pods

In [ ]:
which = [0,1,2,6,7,9] # which digits to work on
data = pods.datasets.decampos_digits(which_digits=which)
Y = data['Y']
labels = data['str_lbls']

You can try to plot some of the digits using `plt.matshow` (the digit
images have size `16x16`).

## Bayesian GPLVM

<span class="editsection-bracket"
style="">\[</span><span class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_gplvm/includes/bayes-gplvm-tutorial.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_gplvm/includes/bayes-gplvm-tutorial.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

In GP-LVM we use a point estimate of the distribution of the input
$\mathbf{Z}$. This estimate is derived through maximum likelihood or
through a maximum a posteriori (MAP) approach. Ideally, we would like to
also estimate a distribution over the input $\mathbf{Z}$. In the
Bayesian GPLVM we approximate the true distribution
$p(\mathbf{Z}|\mathbf{Y})$ by a variational approximation
$q(\mathbf{Z})$ and integrate $\mathbf{Z}$ out (Titsias and Lawrence,
2010).

Approximating the posterior in this way allows us to optimize a lower
bound on the marginal likelihood. Handling the uncertainty in a
principled way allows the model to make an assessment of whether a
particular latent dimension is required, or the variation is better
explained by noise. This allows the algorithm to switch off latent
dimensions. The switching off can take some time though, so below in
Section 6 we provide a pre-learnt module, but to complete section 6
you’ll need to be working in the IPython console instead of the
notebook.

For the moment we’ll run a short experiment applying the Bayesian GP-LVM
with an exponentiated quadratic covariance function.

In [ ]:
Yn = Y

In [ ]:

# Model optimization
input_dim = 5 # How many latent dimensions to use
kern = GPy.kern.RBF(input_dim,ARD=True) # ARD kernel
m = GPy.models.BayesianGPLVM(Yn, input_dim=input_dim, kernel=kern, num_inducing=25)

# initialize noise as 1% of variance in data
#m.likelihood.variance = m.Y.var()/100.
m.optimize(messages=1)

In [ ]:
# Plotting the model
plot_model(m.X.mean, m.rbf.lengthscale.argsort()[:2], labels.flatten())
plt.legend()
m.kern.plot_ARD()
# Saving the model:
m.pickle('digit_bgplvm_rbf.pickle')

Because we are now also considering the uncertainty in the model, this
optimization can take some time. However, you are free to interrupt the
optimization at any point selecting `Kernel->Interupt` from the notepad
menu. This will leave you with the model, `m` in the current state and
you can plot and look into the model parameters.

## Preoptimized Model

A good way of working with latent variable models is to interact with
the latent dimensions, generating data. This is a little bit tricky in
the notebook, so below in section 6 we provide code for setting up an
interactive demo in the standard IPython shell. If you are working on
your own machine you can try this now. Otherwise continue with section
5.

## Multiview Learning: Manifold Relevance Determination

In Manifold Relevance Determination we try to find one latent space,
common for $K$ observed output sets (modalities)
$\{\mathbf{Y}_{k}\}_{k=1}^{K}$. Each modality is associated with a
separate set of ARD parameters so that it switches off different parts
of the whole latent space and, therefore, $\mathbf{Z}$ is softly
segmented into parts that are private to some, or shared for all
modalities. Can you explain what happens in the following example?

Again, you can stop the optimizer at any point and explore the result
obtained with the so far training:

In [ ]:
m = GPy.examples.dimensionality_reduction.mrd_simulation(optimize = False, plot=False)
m.optimize(messages = True, max_iters=3e3, optimizer = 'bfgs')

In [ ]:
_ = m.X.plot()
m.plot_scales()

## Interactive Demo: For Use Outside the Notepad

The module below loads a pre-optimized Bayesian GPLVM model (like the
one you just trained) and allows you to interact with the latent space.
Three interactive figures pop up: the latent space, the ARD scales and a
sample in the output space (corresponding to the current selected latent
point of the other figure). You can sample with the mouse from the
latent space and obtain samples in the output space. You can select
different latent dimensions to vary by clicking on the corresponding
scales with the left and right mouse buttons. This will also cause the
latent space to be projected on the selected latent dimensions in the
other figure.

In [ ]:
from urllib.request import urlopen
import os, sys

In [ ]:
model_path =  'digit_bgplvm_demo.pickle' # local name for model file
status = ""

re = 0
if len(sys.argv) == 2:
    re = 1

if re or not os.path.exists(model_path): # only download the model new, if it was not already
    url = 'http://staffwww.dcs.sheffield.ac.uk/people/M.Zwiessele/gpss/lab3/digit_bgplvm_demo.pickle'
    try:
        with urlopen(url) as u, open(model_path, 'wb') as f:
            file_size = int(u.headers.get('Content-Length') or 0)
            print("Downloading: %s" % (model_path))

            file_size_dl = 0
            block_sz = 8192
            while True:
                buff = u.read(block_sz)
                if not buff:
                    break
                file_size_dl += len(buff)
                f.write(buff)
                sys.stdout.write(" "*(len(status)) + "\r")
                if file_size:
                    status = r"{:7.3f}/{:.3f}MB [{: >7.2%}]".format(file_size_dl/(1.*1e6), file_size/(1.*1e6), float(file_size_dl)/file_size)
                    sys.stdout.write(status)
                    sys.stdout.flush()
            sys.stdout.write(" "*(len(status)) + "\r")
            print(status)
    except Exception as e:
        print("Could not download pre-optimized model:", e)
else:
    print("Already cached, to reload run with 'reload' as the only argument")

In [ ]:
import pickle
try:
    with open('./digit_bgplvm_demo.pickle', 'rb') as f:
        m = pickle.load(f)
except Exception as e:
    print("Could not load pre-optimized model:", e)

Prepare for plotting of this model. If you run on a webserver the
interactive plotting will not work. Thus, you can skip to the next
codeblock and run it on your own machine, later.

In [ ]:
try:
    fig = plt.figure('Latent Space & Scales', figsize=(16,6))
    ax_latent = fig.add_subplot(121)
    ax_scales = fig.add_subplot(122)

    fig_out = plt.figure('Output', figsize=(1,1))
    ax_image  = fig_out.add_subplot(111)
    fig_out.tight_layout(pad=0)

    data_show = GPy.plotting.matplot_dep.visualize.image_show(m.Y[0:1, :], dimensions=(16, 16), transpose=0, invert=0, scale=False, axes=ax_image)
    lvm_visualizer = GPy.plotting.matplot_dep.visualize.lvm_dimselect(m.X.mean.copy(), m, data_show, ax_latent, ax_scales, labels=labels.flatten())
except Exception as e:
    print("Interactive visualizer skipped:", e)

### Observations

Confirm the following observations by interacting with the demo:

-   We tend to obtain more “strange” outputs when sampling from latent
    space areas away from the training inputs.
-   When sampling from the two dominant latent dimensions (the ones
    corresponding to large scales) we differentiate between all digits.
    Also note that projecting the latent space into the two dominant
    dimensions better separates the classes.
-   When sampling from less dominant latent dimensions the outputs vary
    in a more subtle way.

You can also run the dimensionality reduction example

In [ ]:
GPy.examples.dimensionality_reduction.bgplvm_simulation(plot=False)

### Questions

-   Can you see a difference in the ARD parameters to the non Bayesian
    GPLVM?
-   How does the Bayesian GPLVM allow the ARD parameters of the RBF
    kernel magnify the two first dimensions?
-   Is Bayesian GPLVM better in differentiating between different kinds
    of digits?
-   Why does the starting noise variance have to be lower then the
    variance of the observed values?
-   How come we use the lowest variance when using a linear kernel, but
    the highest lengtscale when using an RBF kernel?

## Data for Blastocyst Development in Mice: Single Cell TaqMan Arrays

Now we analyze some single cell data from Guo et al. (2010). Tey
performed qPCR TaqMan array on single cells from the developing
blastocyst in mouse. The data is taken from the early stages of
development when the Blastocyst is forming. At the 32 cell stage the
data is already separated into the trophectoderm (TE) which goes onto
form the placenta and the inner cellular mass (ICM). The ICM further
differentiates into the epiblast (EPI)—which gives rise to the endoderm,
mesoderm and ectoderm—and the primitive endoderm (PE) which develops
into the amniotic sack. Guo et al. (2010) selected 48 genes for
expression measurement. They labelled the resulting cells and their
labels are included as an aide to visualization.

They first visualized their data using principal component analysis. In
the first two principal components this fails to separate the domains.
This is perhaps because the principal components are dominated by the
variation in the 64 cell systems. This in turn may be because there are
more cells from the data set in that regime, and may be because the
natural variation is greater. We first recreate their visualization
using principal component analysis.

In this notebook we will perform PCA on the original data, showing that
the different regimes do not separate.

Next we load in the data. We’ve provided a convenience function for
loading in the data with `pods`. It is loaded in as a `pandas`
DataFrame. This allows us to summarize it with the `describe` attribute.

In [ ]:
import pods

In [ ]:
data = pods.datasets.singlecell()
Y = data['Y']
Y.describe

## Principal Component Analysis

Now we follow Guo et al. (2010) in performing PCA on the data. Rather
than calling a ‘PCA routine’, here we break the algorithm down into its
steps: compute the data covariance, compute the eigenvalues and
eigenvectors and sort according to magnitude of eigenvalue. Because we
want to visualize the data, we’ve chose to compute the eigenvectors of
the *inner product matrix* rather than the covariance matrix. This
allows us to plot the eigenvalues directly. However, this is less
efficient (in this case because the number of genes is smaller than the
number of data) than computing the eigendecomposition of the covariance
matrix.

In [ ]:
import numpy as np

In [ ]:
# obtain a centred version of data.
centredY = Y - Y.mean()
# compute inner product matrix
C = np.dot(centredY,centredY.T)
# perform eigendecomposition
V, U = np.linalg.eig(C)
# sort eigenvalues and vectors according to size
ind = V.argsort()
ev = V[ind[::-1]]
U = U[:, ind[::-1]]

To visualize the result, we now construct a simple helper function. This
will ensure that the plots have the same legends as the GP-LVM plots we
use below.

In [ ]:
import GPy
import matplotlib.pyplot as plt

In [ ]:
def plot_labels(ax, x, y, labels, symbols):
    """A small helper function for plotting with labels"""
    # make sure labels are in order of input:
    ulabels = []
    for lab in labels:
        if not lab in ulabels:
            ulabels.append(lab)
    for i, label in enumerate(ulabels):
        symbol = symbols[i % len(symbols)]
        ind = labels == label
        ax.plot(x[ind], y[ind], symbol)
    ax.legend(ulabels)

## PCA Result

Now, using the helper function we can plot the results with appropriate
labels.

In [ ]:
import matplotlib.pyplot as plt
import mlai.plot as plot
import mlai

In [ ]:
fig, ax = plt.subplots(figsize=plot.big_figsize)
plot_labels(ax, U[:, 0], U[:, 1], data['labels'], '<>^vsd')

mlai.write_figure('singlecell-data-pca.svg', directory='./datasets')

<img src="https://mlatcl.github.io/gpss/slides/diagrams/datasets/singlecell-data-pca.svg" class="" width="60%" style="vertical-align:middle;">

Figure: <i>First two principal compoents of the Guo et al. (2010)
blastocyst development data.</i>

## Bayesian GP-LVM

<span class="editsection-bracket"
style="">\[</span><span class="editsection"
style=""><a href="https://github.com/lawrennd/snippets/edit/main/_gplvm/includes/singlecell-bayes-gplvm.md" target="_blank" onclick="ga('send', 'event', 'Edit Page', 'Edit', 'https://github.com/lawrennd/snippets/edit/main/_gplvm/includes/singlecell-bayes-gplvm.md', 13);">edit</a></span><span class="editsection-bracket" style="">\]</span>

Here we show the new code that uses the Bayesian GP-LVM to fit the data.
This means we can automatically determine the dimensionality of the
model whilst fitting a non-linear dimensionality reduction. The
approximations we use also mean that it is faster than the original
GP-LVM.

In [ ]:
import GPy

In [ ]:
kernel=GPy.kern.RBF(5,ARD=1)+GPy.kern.Bias(5)
model = GPy.models.BayesianGPLVM(Y.values, 5, num_inducing=15, kernel=kernel)
model.optimize(messages=True)

In [ ]:
import matplotlib.pyplot as plt
import mlai.plot as plot
import mlai

In [ ]:
fig, ax = plt.subplots(figsize=plot.big_figsize)
model.plot_latent(ax=ax, labels=data['labels'], marker='<>^vsd')

mlai.write_figure('singlecell-bayes-gplvm.svg', directory='./gplvm')

<img src="https://mlatcl.github.io/gpss/slides/diagrams/gplvm/singlecell-bayes-gplvm.svg" class="" width="60%" style="vertical-align:middle;">

Figure: <i>Visualisation of the Guo et al. (2010) blastocyst development
data with the Bayesian GP-LVM.</i>

In [ ]:
fig, ax = plt.subplots(figsize=plot.big_wide_figsize)
model.kern.plot_ARD(ax=ax)

mlai.write_figure('singlecell-bayes-gplvm-ard.svg', directory='./gplvm')

<img src="https://mlatcl.github.io/gpss/slides/diagrams/gplvm/singlecell-bayes-gplvm-ard.svg" class="" width="80%" style="vertical-align:middle;">

Figure: <i>The ARD parameters of the Bayesian GP-LVM for the Guo et al.
(2010) blastocyst development data.</i>

This gives a really nice result. Broadly speaking two latent dimensions
dominate the representation. When we visualize using these two
dimensions we can see the entire cell phylogeny laid out nicely in the
two dimensions.

## Thanks!

For more information on these subjects and more you might want to check
the following resources.

-   company: [Trent AI](https://trent.ai)
-   book: [The Atomic
    Human](https://www.penguin.co.uk/books/455130/the-atomic-human-by-lawrence-neil-d/9780241625248)
-   twitter: [@lawrennd](https://twitter.com/lawrennd)
-   podcast: [The Talking Machines](http://thetalkingmachines.com)
-   newspaper: [Guardian Profile
    Page](http://www.theguardian.com/profile/neil-lawrence)
-   blog:
    [http://inverseprobability.com](http://inverseprobability.com/blog.html)

## References

Damianou, A., 2015. Deep Gaussian processes and variational propagation
of uncertainty (PhD thesis). University of Sheffield.

Damianou, A., Titsias, M.K., Lawrence, N.D., 2016. Variational inference
for latent variables and uncertain inputs in Gaussian processes. Journal
of Machine Learning Research 17, 1–62.

de Campos, T.E., Babu, B.R., Varma, M., 2009. Character recognition in
natural images, in: Proceedings of the Fourth International Conference
on Computer Vision Theory and Applications - Volume 2: VISAPP,
(VISIGRAPP 2009). INSTICC; SciTePress, pp. 273–280.
<https://doi.org/10.5220/0001770102730280>

Ek, C.H., Rihan, J., Torr, P.H.S., Rogez, G., Lawrence, N.D., 2008a.
Ambiguity modeling in latent spaces, in: Popescu-Belis, A.,
Stiefelhagen, R. (Eds.), Machine Learning for Multimodal Interaction
(MLMI 2008), LNCS. Springer-Verlag, pp. 62–73.

Ek, C.H., Torr, P.H.S., Lawrence, N.D., 2008b. Gaussian process latent
variable models for human pose estimation, in: Popescu-Belis, A.,
Renals, S., Bourlard, H. (Eds.), Machine Learning for Multimodal
Interaction (MLMI 2007), LNCS. Springer-Verlag, Brno, Czech Republic,
pp. 132–143. <https://doi.org/10.1007/978-3-540-78155-4_12>

Guo, G., Huss, M., Tong, G.Q., Wang, C., Sun, L.L., Clarke, N.D.,
Robsonemail, P., 2010. Resolution of cell fate decisions revealed by
single-cell gene expression analysis from zygote to blastocyst.
Developmental Cell 18, 675–685.
<https://doi.org/10.1016/j.devcel.2010.02.012>

Klami, A., Kaski, S., n.d. Local dependent components analysis.

Klami, A., Kaski, S., 2008. Probabilistic approach to detecting
dependencies between data sets. Neurocomputing 72, 39–46.

Leen, G., Fyfe, C., 2006. A Gaussian process latent variable model
formulation of canonical correlation analysis, in: European Symposium on
Artificial Neural Networks. Bruges (Belgium).

Navaratnam, R., Fitzgibbon, A., Cipolla, R., 2007. The joint manifold
model for semi-supervised multi-valued regression, in: IEEE
International Conference on Computer Vision (ICCV). IEEE Computer
Society Press.

Salimbeni, H., Deisenroth, M., 2017. [Doubly stochastic variational
inference for deep Gaussian
processes](http://papers.nips.cc/paper/7045-doubly-stochastic-variational-inference-for-deep-gaussian-processes.pdf),
in: Guyon, I., Luxburg, U.V., Bengio, S., Wallach, H., Fergus, R.,
Vishwanathan, S., Garnett, R. (Eds.), Advances in Neural Information
Processing Systems 30. Curran Associates, Inc., pp. 4591–4602.

Shon, A.P., Grochow, K., Hertzmann, A., Rao, R.P.N., n.d. Learning
shared latent structure for image synthesis and robotic imitation.

Titsias, M.K., Lawrence, N.D., 2010. Bayesian Gaussian process latent
variable model. pp. 844–851.

Tucker, L.R., 1958. An inter-battery method of factor analysis.
Psychometrika 23, 111–136.

Virtanen, S., Klami, A., Kaski, S., n.d. Bayesian CCA via group
sparsity.